In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv(r'C:\Infoct_project 1 sample practice\dataset\predictive_maintenance.csv')
print("Dataset loaded!")
print(df.shape)

Dataset loaded!
(10000, 10)


In [3]:
# Week 2 external context
np.random.seed(42)
df['Ambient_temperature'] = np.random.normal(loc=295, scale=3, size=len(df))
df['Load_density'] = np.random.uniform(low=0.3, high=1.0, size=len(df))

# Week 1 engineered features
df["temp_difference"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["power"] = df["Rotational speed [rpm]"] * df["Torque [Nm]"]
df["tool_wear_rate"] = df["Tool wear [min]"] / df["Rotational speed [rpm]"]
df["heat_stress_index"] = df["Air temperature [K]"] * df["Tool wear [min]"]
df["speed_torque_ratio"] = df["Rotational speed [rpm]"] / df["Torque [Nm]"]
df["wear_per_rotation"] = df["Tool wear [min]"] / df["Rotational speed [rpm]"] * 1000

# Week 2 fusion features
df["temperature_gap"] = df["Air temperature [K]"] - df["Ambient_temperature"]
df["load_stress"] = df["power"] * df["Load_density"]

# Week 3 Day 1 new features
df["thermal_stress"] = df["temp_difference"] * df["tool_wear_rate"]
df["power_per_temp"] = df["power"] / df["Air temperature [K]"]
df["wear_heat_ratio"] = df["tool_wear_rate"] / df["heat_stress_index"].replace(0, 1)

print("All features recreated!")
print(df.shape)

All features recreated!
(10000, 23)


In [10]:
all_features = ['temp_difference', 'power', 'tool_wear_rate','heat_stress_index', 'speed_torque_ratio', 'wear_per_rotation','temperature_gap', 'load_stress','thermal_stress', 'power_per_temp', 'wear_heat_ratio']
print("=== All Features Correlation with Target ===")
correlations = {}
for feature in all_features:
    correlation = df[feature].corr(df['Target'])
    correlations[feature] = correlation
    print(f"{feature}: {correlation:.3f}")

=== All Features Correlation with Target ===
temp_difference: -0.112
power: 0.176
tool_wear_rate: 0.130
heat_stress_index: 0.106
speed_torque_ratio: 0.063
wear_per_rotation: 0.130
temperature_gap: 0.059
load_stress: 0.088
thermal_stress: 0.108
power_per_temp: 0.172
wear_heat_ratio: 0.074


In [7]:
print("=== Feature Selection ===")
strong_features = []
weak_features = []

for feature in all_features:
    corr = abs(df[feature].corr(df['Target']))
    if corr >= 0.10:
        print(f" KEEP - {feature}: {corr:.3f}")
        strong_features.append(feature)
    else:
        print(f" REMOVE - {feature}: {corr:.3f}")
        weak_features.append(feature)

print(f"\nTotal kept: {len(strong_features)}")
print(f"Total removed: {len(weak_features)}")

=== Feature Selection ===
 KEEP - temp_difference: 0.112
 KEEP - power: 0.176
 KEEP - tool_wear_rate: 0.130
 KEEP - heat_stress_index: 0.106
 REMOVE - speed_torque_ratio: 0.063
 KEEP - wear_per_rotation: 0.130
 REMOVE - temperature_gap: 0.059
 REMOVE - load_stress: 0.088
 KEEP - thermal_stress: 0.108
 KEEP - power_per_temp: 0.172
 REMOVE - wear_heat_ratio: 0.074

Total kept: 7
Total removed: 4


In [9]:
final_features = ['Air temperature [K]', 'Process temperature [K]','Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]'] + strong_features
print("=== Final Feature List for LightGBM ===")
print(f"Total features: {len(final_features)}")
for i, feature in enumerate(final_features, 1):
    print(f"{i}. {feature}")

=== Final Feature List for LightGBM ===
Total features: 12
1. Air temperature [K]
2. Process temperature [K]
3. Rotational speed [rpm]
4. Torque [Nm]
5. Tool wear [min]
6. temp_difference
7. power
8. tool_wear_rate
9. heat_stress_index
10. wear_per_rotation
11. thermal_stress
12. power_per_temp
